In [ ]:
import pandas as pd
from tqdm.auto import trange
import random
import math

def get_block_coords(N, row, col):
    """Helper to get the top-left coordinate of the 3x3 block."""
    block_size = int(math.sqrt(N))
    block_row_start = (row // block_size) * block_size
    block_col_start = (col // block_size) * block_size
    return block_row_start, block_col_start

def fitness(board, N):
    violations = 0

    # Check rows
    for r in range(N):
        seen = set()
        for c in range(N):
            val = board[r][c]
            if val != 0:
                if val in seen:
                    violations += 1
                seen.add(val)

    # Check columns
    for c in range(N):
        seen = set()
        for r in range(N):
            val = board[r][c]
            if val != 0:
                if val in seen:
                    violations += 1
                seen.add(val)

    # Check 3x3 blocks (if N is 9) or 2x2 blocks (if N is 4)
    block_size = int(math.sqrt(N))
    for br_start in range(0, N, block_size):
        for bc_start in range(0, N, block_size):
            seen = set()
            for r in range(br_start, br_start + block_size):
                for c in range(bc_start, bc_start + block_size):
                    val = board[r][c]
                    if val != 0:
                        if val in seen:
                            violations += 1
                        seen.add(val)
    return violations

def particle_swarm_optimization(initial_board, N, iterations, swarm_size, c1, c2, w):
    # Store the initial fixed values
    fixed_cells = []
    for r in range(N):
        for c in range(N):
            if initial_board[r][c] != 0:
                fixed_cells.append((r, c))

    def initialize_board_with_random_fill(board):
        new_board = [row[:] for row in board]
        empty_cells = []
        for r in range(N):
            for c in range(N):
                if new_board[r][c] == 0:
                    empty_cells.append((r, c))

        # Fill empty cells with random numbers (1 to N)
        for r, c in empty_cells:
            new_board[r][c] = random.randint(1, N)
        return new_board

    # Particle represents a filled Sudoku board
    class Particle:
        def __init__(self, initial_board_template):
            self.board = initialize_board_with_random_fill(initial_board_template)
            self.pbest_board = [row[:] for row in self.board]
            self.pbest_fitness = fitness(self.board, N)

    swarm = [Particle(initial_board) for _ in range(swarm_size)]

    gbest_board = None
    gbest_fitness = float('inf')

    # Find initial gbest
    for particle in swarm:
        if particle.pbest_fitness < gbest_fitness:
            gbest_fitness = particle.pbest_fitness
            gbest_board = [row[:] for row in particle.pbest_board]

    if gbest_fitness == 0:
        return True, 0, gbest_board # Already solved

    # Main PSO loop
    for iter_count in range(iterations):
        for particle in swarm:
            current_board = [row[:] for row in particle.board]
            empty_cells_coords = [(r, c) for r in range(N) for c in range(N) if (r, c) not in fixed_cells]

            if not empty_cells_coords: # If board is full and no empty cells to change
                continue

            # Simplified "velocity" and "position" update:
            # Perturb the board by swapping values in a few random empty cells.
            # The number of swaps is influenced by the PSO parameters (c1, c2, w) for a more dynamic perturbation.
            # This is a heuristic approximation of PSO's update mechanism for discrete problems.
            num_perturbations = max(1, min(len(empty_cells_coords) // 2,
                                          int(swarm_size * (c1 + c2 + w) / (N*N)) + 1))

            for _ in range(num_perturbations):
                if len(empty_cells_coords) < 2: # Need at least two cells to swap
                    break
                idx1, idx2 = random.sample(range(len(empty_cells_coords)), 2)
                r1, c1_coord = empty_cells_coords[idx1]
                r2, c2_coord = empty_cells_coords[idx2]
                # Swap values
                current_board[r1][c1_coord], current_board[r2][c2_coord] = \
                    current_board[r2][c2_coord], current_board[r1][c1_coord]

            # Also, randomly change a value in an empty cell sometimes
            if empty_cells_coords and random.random() < 0.2 * (c1 + c2):
                 r, c = random.choice(empty_cells_coords)
                 current_board[r][c] = random.randint(1, N)

            new_fitness = fitness(current_board, N)

            # Update pbest
            if new_fitness < particle.pbest_fitness:
                particle.pbest_fitness = new_fitness
                particle.pbest_board = [row[:] for row in current_board]

            particle.board = [row[:] for row in current_board] # Update particle's current position

        # Update gbest after all particles have moved and updated their pbest
        for particle in swarm:
            if particle.pbest_fitness < gbest_fitness:
                gbest_fitness = particle.pbest_fitness
                gbest_board = [row[:] for row in particle.pbest_board]

        if gbest_fitness == 0:
            return True, iter_count + 1, gbest_board

    return False, iterations, gbest_board


# Define the 4x4 Sudoku board
sudoku_4x4 = [
    [1, 2, 0, 0],
    [3, 4, 0, 0],
    [0, 0, 1, 2],
    [0, 0, 3, 4]
]

# Define the 9x9 Sudoku board (a classic example with some empty cells)
sudoku_9x9 = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

# PSO Parameters for 4x4
swarm_sizes_pso_4x4 = [10, 20]
c1_values_pso = [0.5, 1.0]
c2_values_pso = [1.5, 2.0]
w_values_pso = [0.7, 0.9]
repeats_pso = 10 # Number of times to repeat each experiment configuration
max_iterations_pso_4x4 = 1000

# PSO Parameters for 9x9 (potentially different ranges for a larger problem)
swarm_sizes_pso_9x9 = [50, 100]
c1_values_pso_9x9 = [0.5, 1.0]
c2_values_pso_9x9 = [1.5, 2.0]
w_values_pso_9x9 = [0.7, 0.9]
max_iterations_pso_9x9 = 1000

In [ ]:
def run_experiment_pso(sudoku_board, N, swarm_sizes, c1_values, c2_values, w_values, repeats=1, max_iterations=2000):
    results = []

    for swarm_size in swarm_sizes:
        for c1 in c1_values:
            for c2 in c2_values:
                for w in w_values:
                    success_count = 0
                    total_iterations = 0
                    best_overall_board = None
                    for r in trange(repeats, desc=f"PSO N={N}, Swarm={swarm_size}, C1={c1}, C2={c2}, W={w}", leave=False):
                        solved, iters, final_board = particle_swarm_optimization(
                            initial_board=sudoku_board, N=N, iterations=max_iterations,
                            swarm_size=swarm_size, c1=c1, c2=c2, w=w
                        )
                        if solved:
                            success_count += 1
                            print(f"Solved N={N} with Swarm={swarm_size}, C1={c1}, C2={c2}, W={w} in {iters} iterations (Repeat {r+1}/{repeats}):")
                            print(final_board)
                            if best_overall_board is None or fitness(final_board, N) < fitness(best_overall_board, N):
                                best_overall_board = final_board
                        total_iterations += iters
                        if not solved and (best_overall_board is None or fitness(final_board, N) < fitness(best_overall_board, N)):
                            best_overall_board = final_board

                    results.append({
                        "Lentelės dydis": f"{N}x{N}",
                        "Swarm Size": swarm_size,
                        "C1": c1,
                        "C2": c2,
                        "W": w,
                        "Sėkmės dažnis": f"{success_count}/{repeats}",
                        "Vid. iteracijų": round(total_iterations / repeats, 2)
                    })
                    if success_count == 0 and best_overall_board is not None:
                        print(f"For N={N}, Swarm={swarm_size}, C1={c1}, C2={c2}, W={w}: No solution found in any repeat. Best board found (fitness {fitness(best_overall_board, N)}):\n")
                        print(best_overall_board)
    return pd.DataFrame(results)

In [ ]:
import pandas as pd

print("\n--- Running Particle Swarm Optimization Experiments ---\n")
pd.set_option('display.max_columns', None) # Ensure all columns are displayed

df_pso_4x4 = run_experiment_pso(sudoku_4x4, 4, swarm_sizes_pso_4x4, c1_values_pso, c2_values_pso, w_values_pso, repeats_pso, max_iterations_pso_4x4)
df_pso_9x9 = run_experiment_pso(sudoku_9x9, 9, swarm_sizes_pso_9x9, c1_values_pso_9x9, c2_values_pso_9x9, w_values_pso_9x9, repeats_pso, max_iterations_pso_9x9)

display(df_pso_4x4.head(6))
display(df_pso_9x9.head(6))


--- Running Particle Swarm Optimization Experiments ---



PSO N=4, Swarm=10, C1=0.5, C2=1.5, W=0.7:   0%|          | 0/10 [00:00<?, ?it/s]

Solved N=4 with Swarm=10, C1=0.5, C2=1.5, W=0.7 in 111 iterations (Repeat 6/10):
[[1, 2, 4, 3], [3, 4, 2, 1], [4, 3, 1, 2], [2, 1, 3, 4]]


PSO N=4, Swarm=10, C1=0.5, C2=1.5, W=0.9:   0%|          | 0/10 [00:00<?, ?it/s]

Solved N=4 with Swarm=10, C1=0.5, C2=1.5, W=0.9 in 831 iterations (Repeat 10/10):
[[1, 2, 4, 3], [3, 4, 2, 1], [4, 3, 1, 2], [2, 1, 3, 4]]


...

For N=9, Swarm=100, C1=1.0, C2=2.0, W=0.7: No solution found in any repeat. Best board found (fitness 50):

[[5, 3, 4, 2, 7, 5, 4, 5, 6], [6, 1, 7, 1, 9, 5, 3, 9, 2], [5, 9, 8, 3, 4, 6, 7, 6, 4], [8, 6, 9, 5, 6, 3, 2, 1, 3], [4, 6, 6, 8, 2, 3, 9, 4, 1], [7, 5, 5, 4, 2, 7, 5, 8, 6], [7, 6, 3, 9, 5, 5, 2, 8, 1], [4, 1, 2, 4, 1, 9, 8, 3, 5], [3, 9, 5, 6, 8, 2, 4, 7, 9]]


PSO N=9, Swarm=100, C1=1.0, C2=2.0, W=0.9:   0%|          | 0/10 [00:00<?, ?it/s]

For N=9, Swarm=100, C1=1.0, C2=2.0, W=0.9: No solution found in any repeat. Best board found (fitness 54):

[[5, 3, 6, 7, 7, 8, 3, 1, 7], [6, 4, 1, 1, 9, 5, 8, 7, 8], [2, 9, 8, 5, 2, 3, 5, 6, 2], [8, 2, 7, 1, 6, 4, 4, 5, 3], [4, 2, 9, 8, 5, 3, 3, 6, 1], [7, 5, 1, 9, 2, 5, 3, 8, 6], [4, 6, 3, 9, 3, 1, 2, 8, 1], [3, 9, 2, 4, 1, 9, 8, 7, 5], [1, 1, 1, 2, 8, 6, 1, 7, 9]]


,Lentelės dydis,Swarm Size,C1,C2,W,Sėkmės dažnis,Vid. iteracijų
0,4x4,10,0.5,1.5,0.7,1/10,911.1
1,4x4,10,0.5,1.5,0.9,1/10,983.1
2,4x4,10,0.5,2.0,0.7,3/10,884.1
3,4x4,10,0.5,2.0,0.9,2/10,862.8
4,4x4,10,1.0,1.5,0.7,1/10,942.5
5,4x4,10,1.0,1.5,0.9,1/10,943.6


,Lentelės dydis,Swarm Size,C1,C2,W,Sėkmės dažnis,Vid. iteracijų
0,9x9,50,0.5,1.5,0.7,0/10,1000.0
1,9x9,50,0.5,1.5,0.9,0/10,1000.0
2,9x9,50,0.5,2.0,0.7,0/10,1000.0
3,9x9,50,0.5,2.0,0.9,0/10,1000.0
4,9x9,50,1.0,1.5,0.7,0/10,1000.0
5,9x9,50,1.0,1.5,0.9,0/10,1000.0
